### stg_validation_theirstack

In [3]:
import os
import re
import json
import pandas as pd
import snowflake.connector
from dotenv import load_dotenv
from IPython.display import display

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 100)

load_dotenv()

conn = snowflake.connector.connect(
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    role=os.getenv("SNOWFLAKE_ROLE"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database=os.getenv("SNOWFLAKE_DATABASE"),
)

def run_query(sql: str) -> pd.DataFrame:
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [desc[0] for desc in cur.description]
    cur.close()
    return pd.DataFrame(rows, columns=cols)

print("Connected!")

Connected!


In [4]:
# ── stg_theirstack ────────────────────────────────────────────────────────────
raw_df = run_query("SELECT SOURCE, RAW_PAYLOAD::STRING AS payload, INGESTED_AT FROM RAW.THEIRSTACK.SRC_POSTINGS")
print(f"Raw rows: {len(raw_df)}")
print(f"Columns: {raw_df.columns.tolist()}")

parsed = []
for _, row in raw_df.iterrows():
    p = json.loads(row['PAYLOAD'])
    parsed.append({
        # --- identity ---
        'job_id':           str(p.get('id')),
        'source_raw':       row['SOURCE'],
        'ingested_at':      row['INGESTED_AT'],

        # --- core fields ---
        'job_title':        p.get('job_title'),
        'company_name':     p.get('company'),
        'job_url':          p.get('url'),
        'date_posted':      p.get('date_posted'),
        'description':      p.get('description'),

        # --- location ---
        'city':             p.get('short_location', '').split(',')[0].strip() if p.get('short_location') else None,
        'state':            p.get('state_code'),
        'country':          p.get('country_code'),
        'latitude':         p.get('latitude'),
        'longitude':        p.get('longitude'),

        # --- employment ---
        'employment_type_raw': p.get('employment_statuses'),

        # --- work model (reliable booleans) ---
        'remote_raw':       p.get('remote'),
        'hybrid_raw':       p.get('hybrid'),

        # --- salary (already annual USD) ---
        'salary_min_raw':   p.get('min_annual_salary_usd'),
        'salary_max_raw':   p.get('max_annual_salary_usd'),
    })

stg_ts = pd.DataFrame(parsed)
print(f"Parsed rows: {len(stg_ts)}")

Raw rows: 18
Columns: ['SOURCE', 'PAYLOAD', 'INGESTED_AT']
Parsed rows: 18


In [5]:
# Normalize source
stg_ts['source'] = 'theirstack'

# Normalize date
stg_ts['date_posted'] = pd.to_datetime(stg_ts['date_posted'], utc=True).dt.date

# Employment type — TheirStack returns a list e.g. ['full_time']
def normalize_ts_employment(val) -> str:
    if not val or not isinstance(val, list):
        return None
    v = val[0].lower() if val else None
    if not v:
        return None
    if 'full' in v:    return 'full_time'
    if 'part' in v:    return 'part_time'
    if 'contract' in v: return 'contract'
    return 'other'

stg_ts['employment_type'] = stg_ts['employment_type_raw'].apply(normalize_ts_employment)

# Work model — remote and hybrid booleans are reliable on TheirStack
def derive_ts_work_model(row) -> str:
    if row['remote_raw']:  return 'remote'
    if row['hybrid_raw']:  return 'hybrid'
    return 'onsite'

stg_ts['work_model'] = stg_ts.apply(derive_ts_work_model, axis=1)

# Salary — already annual USD, no filtering needed
stg_ts['salary_min'] = stg_ts['salary_min_raw']
stg_ts['salary_max'] = stg_ts['salary_max_raw']

print("employment_type value counts:")
print(stg_ts['employment_type'].value_counts(dropna=False))
print("\nwork_model value counts:")
print(stg_ts['work_model'].value_counts(dropna=False))
print("\nSalary coverage:")
print(f"  salary_min populated: {stg_ts['salary_min'].notna().sum()} / {len(stg_ts)}")

employment_type value counts:
employment_type
full_time    15
contract      2
NaN           1
Name: count, dtype: int64

work_model value counts:
work_model
onsite    9
hybrid    7
remote    2
Name: count, dtype: int64

Salary coverage:
  salary_min populated: 2 / 18


In [7]:
# Senior title filter — same regex as JSearch
SENIOR_RE = re.compile(
    r'\b(senior|sr\.?|lead|principal|staff|manager|director|vp|vice president|'
    r'avp|head of|architect|chief|svp|evp|gvp|president|officer|executive|leader)\b',
    re.IGNORECASE
)

stg_ts['is_senior'] = stg_ts['job_title'].apply(
    lambda t: bool(SENIOR_RE.search(t)) if t else False
)

print(f"Senior titles flagged: {stg_ts['is_senior'].sum()} / {len(stg_ts)}")
print(stg_ts[stg_ts['is_senior']]['job_title'].tolist())

Senior titles flagged: 0 / 18
[]


In [8]:
# Dedup within TheirStack (shouldn't be any but good hygiene)
before = len(stg_ts)
stg_ts = (
    stg_ts.sort_values('ingested_at', ascending=False)
          .drop_duplicates(subset='job_id', keep='first')
          .reset_index(drop=True)
)
print(f"Rows before dedup: {before} | after: {len(stg_ts)} | removed: {before - len(stg_ts)}")

Rows before dedup: 18 | after: 18 | removed: 0


In [10]:
# Apply filters and select final columns
stg_ts_final = (
    stg_ts[~stg_ts['is_senior']]
    [[
        'job_id', 'source', 'job_title', 'company_name', 'job_url',
        'date_posted', 'description', 'city', 'state', 'country',
        'latitude', 'longitude', 'work_model', 'employment_type',
        'salary_min', 'salary_max', 'ingested_at',
    ]]
    .reset_index(drop=True)
)

print(f"Final stg_theirstack rows: {len(stg_ts_final)}")
print(f"\nNull rates:")
print((stg_ts_final.isna().sum() / len(stg_ts_final) * 100).round(1).to_string())
print(f"\nSample output:")
display(stg_ts_final.head(10))

Final stg_theirstack rows: 18

Null rates:
job_id              0.0
source              0.0
job_title           0.0
company_name        0.0
job_url             0.0
date_posted         0.0
description         0.0
city                0.0
state               0.0
country             0.0
latitude            0.0
longitude           0.0
work_model          0.0
employment_type     5.6
salary_min         88.9
salary_max         88.9
ingested_at         0.0

Sample output:


,job_id,source,job_title,company_name,job_url,date_posted,description,city,state,country,latitude,longitude,work_model,employment_type,salary_min,salary_max,ingested_at
0,704913940,theirstack,Data Analyst-Institute for Advanced Medicine-Aids Center Peter Krueger Clini...,Mount Sinai Morningside,https://www.linkedin.com/jobs/view/data-analyst-institute-for-advanced-medic...,2026-06-03,**Description**\nJob Description\n \n \n**Position Title**\n**Data Analyst...,New York,NY,US,40.714270,-74.00597,onsite,full_time,NaN,NaN,2026-06-04 12:04:35.529034+00:00
1,704438351,theirstack,"Data Analyst, Center of Surgical and Transplant Applied Research (CSTAR)",NYU Langone Health,https://www.linkedin.com/jobs/view/data-analyst-center-of-surgical-and-trans...,2026-06-03,1153308\_RR00113253 Job ID: 1153308\_RR00113253\n \n \nNYU Grossman School...,New York,NY,US,40.714270,-74.00597,onsite,full_time,NaN,NaN,2026-06-04 12:04:35.529034+00:00
2,702956448,theirstack,GIS Data Analyst,Trident Consulting,https://www.linkedin.com/jobs/view/gis-data-analyst-at-trident-consulting-44...,2026-06-02,Trident Consulting is seeking a “\n**GIS Data Analyst**\n” for one of our cl...,New York,NY,US,40.714270,-74.00597,onsite,contract,NaN,NaN,2026-06-04 12:04:35.529034+00:00
3,703117190,theirstack,"Analytics Engineer, Service Ops Analytics & AI",Capgemini,https://www.linkedin.com/jobs/view/analytics-engineer-service-ops-analytics-...,2026-06-02,The goal of analytics engineering team within the Service Analytics and AI o...,New York,NY,US,40.714270,-74.00597,onsite,full_time,NaN,NaN,2026-06-04 12:04:35.529034+00:00
4,704952793,theirstack,Data Analyst,ATC,https://www.linkedin.com/jobs/view/data-analyst-at-atc-4424220166,2026-06-03,**About Us:**\n\nAmerican Technology Consulting (ATC) is a service-first tec...,New York,NY,US,40.714270,-74.00597,onsite,full_time,NaN,NaN,2026-06-04 12:04:35.529034+00:00
5,705349326,theirstack,Economic Data Analyst,Toll International LLC,https://www.linkedin.com/jobs/view/economic-data-analyst-at-toll-internation...,2026-06-03,Help shape the economic insights that support some of the most critical tran...,New York,NY,US,40.714270,-74.00597,hybrid,full_time,NaN,NaN,2026-06-04 12:04:35.529034+00:00
6,704607328,theirstack,Data Analyst,Hanvok Group LLC,https://www.linkedin.com/jobs/view/data-analyst-at-hanvok-group-llc-4424091714,2026-06-03,**Company Description**\nHanvok Group LLC helps professionals and businesses...,New York,NY,US,40.714270,-74.00597,remote,full_time,NaN,NaN,2026-06-04 12:04:35.529034+00:00
7,703949131,theirstack,Data Analyst,ThesisGrid,http://www.indeed.com/job/data-analyst-1ace763e7e33acce,2026-06-03,About ThesisGrid Capital\n\nThesisGrid Capital is an AI-native investment re...,New York,NY,US,40.762188,-73.97265,onsite,full_time,70000.0,120000.0,2026-06-04 12:04:35.529034+00:00
8,702699905,theirstack,Data Analyst,edkey,https://www.linkedin.com/jobs/view/data-analyst-at-edkey-4423957473,2026-06-02,**About the Role**\n\nWe are hiring a Data Analyst on the Product Data Scien...,New York,NY,US,40.714270,-74.00597,hybrid,full_time,NaN,NaN,2026-06-04 12:04:35.529034+00:00
9,704833094,theirstack,Data Analyst (Entry-Level / Junior),Africa Explorer,https://www.linkedin.com/jobs/view/data-analyst-entry-level-junior-at-africa...,2026-06-03,**Data Analyst (Entry-Level / Junior)**\n\nRole Description\n\nAnalyze and i...,New York,NY,US,40.714270,-74.00597,hybrid,full_time,NaN,NaN,2026-06-04 12:04:35.529034+00:00


In [11]:
print(f"\nAll titles:")
for t in sorted(stg_ts_final['job_title'].tolist()):
    print(t)


All titles:
Advertising Data Analyst
Analytics Engineer, Service Ops Analytics & AI
Data Analyst
Data Analyst
Data Analyst
Data Analyst
Data Analyst
Data Analyst (Entry-Level / Junior)
Data Analyst (Entry-Level / Junior)
Data Analyst, Center of Surgical and Transplant Applied Research (CSTAR)
Data Analyst-Institute for Advanced Medicine-Aids Center Peter Krueger Clinic-Mount Sinai Health System-Full Time-Days
Economic Data Analyst
Economic Data Analyst (Transportation & Infrastructure)
GIS Data Analyst
Junior Data Analyst w/ Workday Exp....Rate-$30/hronC2C....Locals Only
Market Data Analyst
Product Data Analyst
SQL Data Analyst


In [ ]:
conn.close()
print('Connection closed.')